# Ch 6. BPE 토크나이저 직접 훈련

**BPE 알고리즘 핵심을 이해하고 HuggingFace `tokenizers` 라이브러리로 8K vocab BPE를 직접 훈련한다.**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/desty/study-tiny-llm/blob/main/notebooks/part2/ch06_bpe.ipynb)

In [ ]:
# !pip install -q tokenizers datasets

import torch
print(f"PyTorch: {torch.__version__}")

try:
    import tokenizers
    print(f"tokenizers: {tokenizers.__version__}")
except ImportError:
    print("tokenizers not installed — run: pip install tokenizers")

## BPE 알고리즘 한 페이지

**Byte-Pair Encoding** (Sennrich et al., 2016이 NMT에 도입).

```
초기:    각 글자 = 한 토큰
반복:
  1. 텍스트에서 가장 자주 나오는 인접 쌍 찾기
  2. 그 쌍을 하나의 새 토큰으로 합치기
  3. vocab에 추가
종료:    원하는 vocab size 도달
```

자주 나오는 substring이 **하나의 토큰**이 되고, 드문 단어는 **여러 작은 토큰**으로 쪼개진다.

In [ ]:
# BPE 알고리즘 직접 구현 (교육용 — 간단 버전)
from collections import Counter

def get_pairs(vocab):
    """vocab에서 인접 심볼 쌍과 빈도 계산."""
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

def merge_vocab(pair, vocab):
    """vocab에서 pair를 합쳐 새 vocab 반환."""
    new_vocab = {}
    bigram = " ".join(pair)
    replacement = "".join(pair)
    for word, freq in vocab.items():
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = freq
    return new_vocab

# 간단한 corpus
vocab = {
    "l o w </w>": 5,
    "l o w e s t </w>": 2,
    "n e w e r </w>": 6,
    "w i d e r </w>": 3,
    "n e w </w>": 2,
}

print("BPE 병합 단계:")
print(f"초기 vocab: {vocab}\n")

n_merges = 5
for i in range(n_merges):
    pairs = get_pairs(vocab)
    best = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)
    print(f"단계 {i+1}: '{best[0]} {best[1]}' -> '{''.join(best)}' (빈도: {pairs[best]})")

print(f"\n최종 vocab: {vocab}")

## 최소 예제 — 8K BPE 30초 훈련

**관찰**:
- `Ġ` = ByteLevel BPE의 띄어쓰기 표시
- 한국어는 학습 데이터에 없어 **byte 단위로 분해** → 토큰 수 폭발
- 같은 문장이 영어 4 토큰 vs 한국어 18 토큰

In [ ]:
# train_bpe.py
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from tokenizers.processors import ByteLevel as ByteLevelProcessor
from datasets import load_dataset

# 1. 빈 BPE 토크나이저
tok = Tokenizer(models.BPE())

# 2. Pre-tokenizer — 입력을 byte 단위로
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok.decoder = decoders.ByteLevel()
tok.post_processor = ByteLevelProcessor(trim_offsets=True)

# 3. 학습 corpus iterator — TinyStories
ds = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
def iter_text():
    for i, row in enumerate(ds):
        if i >= 100_000: break  # 10만 동화면 충분
        yield row["text"]

# 4. Trainer 설정
trainer = trainers.BpeTrainer(
    vocab_size=8000,
    special_tokens=["<|endoftext|>", "<|pad|>"],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    show_progress=True,
)

# 5. 훈련 (보통 30초 ~ 2분)
print("BPE 훈련 시작...")
tok.train_from_iterator(iter_text(), trainer=trainer)

# 6. 저장
tok.save("tokenizer.json")
print(f"vocab size: {tok.get_vocab_size()}")

In [ ]:
# check_bpe.py — 토큰화 결과 확인
from tokenizers import Tokenizer
tok = Tokenizer.from_file("tokenizer.json")

texts = [
    "Once upon a time",
    "옛날 옛적에 작은 마을에",
    "Lily loved her toy car",
]
for t in texts:
    enc = tok.encode(t)
    print(f"  '{t}'")
    print(f"    tokens: {enc.tokens}")
    print(f"    count:  {len(enc.ids)}")
    print()

## 실전 — 한국어 합성 데이터 섞어 다시 훈련

캡스톤용 (한국어 동화 생성기) 토크나이저는 한국어 데이터로 학습해야 효율이 살아남.

| 토크나이저 | "옛날 옛적에 작은 마을에" |
|---|---:|
| ByteLevel BPE (영어 학습) | **18 토큰** |
| ByteLevel BPE (한국어 학습) | **6~8 토큰** |
| GPT-4 cl100k_base (다국어) | 9 토큰 |
| Qwen 2.5 BPE (다국어) | 5 토큰 |

In [ ]:
# train_bpe_ko.py — 한국어 합성 데이터가 있는 경우
import os, json
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

# 한국어 합성 동화가 없으면 샘플 데이터로 시연
sample_ko_texts = [
    "옛날 옛적에 토끼 한 마리가 살았어요. 토끼는 당근을 아주 좋아했어요.",
    "작은 마을에 곰 두두가 살았어요. 두두는 매일 아침 꽃을 바라보았어요.",
    "달빛이 환한 밤, 고양이 미미는 친구들과 놀았어요. 모두 행복했어요.",
    "비가 내리는 날, 할머니는 따뜻한 수프를 만들었어요.",
    "나무 아래서 새 한 마리가 노래했어요. 아이들이 모여들었어요.",
] * 200  # 반복해서 최소 corpus 크기 확보

tok_ko = Tokenizer(models.BPE())
tok_ko.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok_ko.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=8000,
    special_tokens=["<|endoftext|>", "<|pad|>"],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
)
tok_ko.train_from_iterator(iter(sample_ko_texts), trainer=trainer)
tok_ko.save("tokenizer_ko.json")
print(f"한국어 토크나이저 vocab size: {tok_ko.get_vocab_size()}")

# 비교
test_text = "옛날 옛적에 작은 마을에"
print(f"\n테스트 문장: '{test_text}'")

tok_en = Tokenizer.from_file("tokenizer.json")
print(f"  영어 BPE 토큰 수: {len(tok_en.encode(test_text).ids)}")
print(f"  한국어 BPE 토큰 수: {len(tok_ko.encode(test_text).ids)}")

In [ ]:
# transformers 호환 래퍼
from transformers import PreTrainedTokenizerFast

fast_tok = PreTrainedTokenizerFast(
    tokenizer_object=tok_ko,
    bos_token="<|endoftext|>",
    eos_token="<|endoftext|>",
    pad_token="<|pad|>",
)
print("transformers PreTrainedTokenizerFast 래핑 성공")
print(f"vocab size: {fast_tok.vocab_size}")

# 저장
fast_tok.save_pretrained("tokenizer_ko_hf")
print("저장 완료: tokenizer_ko_hf/")

## 연습문제

1. 본 책 §4 코드를 그대로 돌려 8K vocab BPE를 훈련하고 한국어 1문장의 토큰 수를 측정하라. §5의 한국어 학습본과 비교.
2. vocab_size를 4K · 8K · 16K · 32K로 바꿔 훈련해보고 같은 100 문서의 평균 토큰 수를 비교. 어디서 수익 체감(diminishing returns)?
3. `tokenizer.encode("123,456원")` 결과를 본 책 BPE와 GPT-4 `tiktoken`으로 비교. 숫자 처리에서 차이는?
4. 한국어 자모 분리 (`unicodedata.normalize('NFD', text)`) 한 다음 BPE 훈련하면 §5 표의 토큰 수가 어떻게 변하나?
5. **(생각해볼 것)** vocab=8K BPE에 한국어 합성 1만 문서로 학습하면 한국어 토큰 효율은 좋아진다. 그 대신 **영어 토큰 효율**은 어떻게 변하는가? 둘 다 잡으려면?

In [ ]:
# 연습문제 1 — 이미 위에서 완료. 추가 비교:
from tokenizers import Tokenizer

tok_en = Tokenizer.from_file("tokenizer.json")
tok_ko = Tokenizer.from_file("tokenizer_ko.json")

test_sentences = [
    "Once upon a time, there was a little girl.",
    "옛날 옛적에 작은 마을에 토끼가 살았어요.",
    "The quick brown fox jumps over the lazy dog.",
    "인공지능 모델을 직접 훈련하는 방법",
]

print(f"{'문장':40s}  {'영어BPE':>8s}  {'한국BPE':>8s}")
print("-" * 62)
for s in test_sentences:
    n_en = len(tok_en.encode(s).ids)
    n_ko = len(tok_ko.encode(s).ids)
    print(f"  {s[:38]:38s}  {n_en:8d}  {n_ko:8d}")

In [ ]:
# 연습문제 2 — vocab_size별 토큰 수 비교
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from datasets import load_dataset

# 100개 문서 캐시
ds = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
corpus = []
for i, row in enumerate(ds):
    if i >= 10_000: break
    corpus.append(row["text"])

test_docs = corpus[:100]
results = []

for vocab_size in [4000, 8000, 16000, 32000]:
    t = Tokenizer(models.BPE())
    t.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    t.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["<|endoftext|>", "<|pad|>"],
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    )
    t.train_from_iterator(iter(corpus), trainer=trainer)
    avg_tokens = sum(len(t.encode(d).ids) for d in test_docs) / len(test_docs)
    results.append((vocab_size, avg_tokens))
    print(f"vocab={vocab_size:5d}: 평균 {avg_tokens:.1f} tokens/doc")

print("\n수익 체감 분석:")
for i in range(1, len(results)):
    v1, t1 = results[i-1]
    v2, t2 = results[i]
    reduction = (t1 - t2) / t1 * 100
    print(f"  {v1}→{v2}: {reduction:.1f}% 감소")

In [ ]:
# 연습문제 3 — 숫자 처리 비교
from tokenizers import Tokenizer

tok_en = Tokenizer.from_file("tokenizer.json")
test_numbers = ["123", "123,456원", "3.14", "2024년", "010-1234-5678"]

print("숫자 토큰화 비교 (본 책 BPE):")
for t in test_numbers:
    enc = tok_en.encode(t)
    print(f"  '{t}' -> {enc.tokens} ({len(enc.ids)} tokens)")

In [ ]:
# 연습문제 4 — NFD 자모 분리 후 BPE
import unicodedata

test_ko = "옛날 옛적에 작은 마을에"
nfd_text = unicodedata.normalize('NFD', test_ko)
print(f"원본: '{test_ko}'  ({len(test_ko)} 글자)")
print(f"NFD:  '{nfd_text}'  ({len(nfd_text)} 코드포인트)")

# NFD로 변환한 corpus로 BPE 훈련
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

nfd_corpus = [unicodedata.normalize('NFD', t) for t in corpus[:5000]]

tok_nfd = Tokenizer(models.BPE())
tok_nfd.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok_nfd.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(
    vocab_size=8000,
    special_tokens=["<|endoftext|>", "<|pad|>"],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
)
tok_nfd.train_from_iterator(iter(nfd_corpus), trainer=trainer)

tok_en_loaded = Tokenizer.from_file("tokenizer.json")

print(f"\n테스트: '{test_ko}'")
print(f"  영어BPE: {len(tok_en_loaded.encode(test_ko).ids)} tokens")
print(f"  NFD BPE: {len(tok_nfd.encode(nfd_text).ids)} tokens")

In [ ]:
# 연습문제 5 — 메모
print("""
생각해볼 것: 한국어 vocab이 많아지면 영어 vocab 슬롯이 줄어든다.

해결책:
1. vocab_size 늘리기 (8K -> 16K 이상)
2. 영어+한국어 균형 있게 섞은 corpus로 훈련
3. 다국어 토크나이저 (SentencePiece) 사용
""")